In [3]:
import pandas as pd
import requests
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
ITAD_API_KEY = "0c3929a700f88f22c3ae0b877c3fa41fcf3720c7"
session = requests.Session()



def fetch_single_game_data(appid):
    """Takes a Steam AppID integer as a parameter, requests data from Steam, returns the game's AppID and Title"""
    details_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&filters=basic"

    try:
        res = session.get(details_url, timeout=5)
        if res.status_code == 200:
            json_data = res.json()
            if (
                json_data
                and str(appid) in json_data
                and json_data[str(appid)].get("success")
            ):
                app_data = json_data[str(appid)]["data"]
                app_type = app_data.get("type", "").lower()

                if app_type == "game":
                    return {
                        "steam_appid": str(appid),
                        "title": app_data.get("name", "")
                    }
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to retrieve Steam data for AppID: {appid}") from e







def get_game_price_history_itad(steam_appid, api_key, since_date="2000-01-01T00:00:00Z"):
    """"Takes a Steam AppID integer as a parameter, requests data from ITAD, returns a dataframe of the target game's price history"""
    lookup_url = "https://api.isthereanydeal.com/games/lookup/v1"
    lookup_res = requests.get(
        lookup_url, params={"key": api_key, "appid": steam_appid}
    ).json()

    game_id = lookup_res.get("game", {}).get("id")
    if not game_id:
        raise ValueError(f"Could not find ITAD game data for Steam AppID: {steam_appid}")

    history_url = "https://api.isthereanydeal.com/games/history/v2"
    params = {
        "key": api_key,
        "id": game_id,
        "shops": 61,
        "since": since_date
    }

    history_res = session.get(history_url, params=params)

    if history_res.status_code != 200:
        raise ValueError(f"No ITAD price history records returned for AppID: {steam_appid}")

    history_data = history_res.json()

    records = []
    for entry in history_data:
        deal = entry.get("deal", {})
        records.append(
            {
                "timestamp_raw": entry.get("timestamp"),
                "date": pd.to_datetime(entry.get("timestamp")),
                "price": deal.get("price", {}).get("amount"),
                "regular_price": deal.get("regular", {}).get("amount"),
            }
        )

    df = pd.DataFrame(records)

    return df

def convert_itad_to_target_format(df_itad):
    """Takes price history dataframe as parameter, returns it properly formatted"""
    df = df_itad.copy()

    df["DateTime"] = (
        pd.to_datetime(df["timestamp_raw"], utc=True)
        .dt.tz_convert(None)
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    df=df.rename(columns={"price":"Final price", "regular_price":"Retail Price"})

    df["Historical Low"] = df["Final price"].cummin()

    df=df[["DateTime","Final price","Historical Low", "Retail Price"]]

    return df



def initial_game_processing(df):
    """Takes price history dataframe as parameter, cleans, sorts and interprets data, then returns it"""
    df=df.rename(columns={"Final price":"Current Price","DateTime":"Date"})
    df["Current Price"]=df["Current Price"].replace(0, np.nan)
    df=df.dropna(how="any").reset_index(drop=True)
    df["Date"]=pd.to_datetime(df["Date"])
    df=df[["Date","Current Price","Historical Low","Retail Price"]].sort_values(by=["Date"])
    discounts = df[df["Current Price"] < df["Retail Price"]].copy()
    discounts["Days Since Discount"]=discounts["Date"].diff(periods=1).dt.days
    discounts = discounts[
    (discounts["Days Since Discount"] > 0) &
    (discounts["Days Since Discount"] <= 730)
    ]
    discounts["Month"]=discounts["Date"].dt.month
    discounts["DayofWeek"]=discounts["Date"].dt.dayofweek
    discounts["DayofYear"]=discounts["Date"].dt.dayofyear
    discounts["Occurrence"] = np.ceil(discounts["Date"].dt.day / 7).astype(int)
    discounts["Days Until Next Sale"]=discounts["Days Since Discount"].shift(-1)
    discounts=discounts.dropna(subset=["Days Until Next Sale"])
    return discounts



def create_target_game_features(raw_game_df, feature_columns):
    """Takes input data columns and target game's price history dataframe as parameters, formats data for the RandomForestRegressor and returns it as new dataframe"""
    historical_low = raw_game_df["Historical Low"].iloc[-1]

    last_sale_date = pd.to_datetime(raw_game_df["DateTime"].iloc[-1])
    now = pd.Timestamp.now()
    live_days_since = (now - last_sale_date).days

    current_features = pd.DataFrame([{
        "Days Since Discount": live_days_since,
        "Historical Low": historical_low,
        "Month": now.month,
        "DayofWeek": now.dayofweek,
        "DayofYear": now.dayofyear,
        "Occurrence": int(np.ceil(now.day / 7))
    }])

    return current_features[feature_columns]








def prediction_function_single(target_appid, print_test_bool):
    """Trains regression and classification models on historical sale data to benchmark model performance and accuracy."""
    raw_game_df = convert_itad_to_target_format(get_game_price_history_itad(target_appid, ITAD_API_KEY))
    if raw_game_df.empty:
        print(f"No price history found for AppID {target_appid}.")
    else:
        discounts = initial_game_processing(raw_game_df)
        discounts = discounts[discounts["Days Since Discount"] > 0]

        if len(discounts) <= 5:
            print(f"Not enough historical sale records to train a model.")
        else:
            feature_cols = [
                "Days Since Discount",
                "Historical Low",
                "Month",
                "DayofWeek",
                "DayofYear",
                "Occurrence",
            ]

            X = discounts[feature_cols]
            Y_reg = discounts["Days Until Next Sale"]


            X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
                X, Y_reg, test_size=0.2, random_state=42
            )

            reg_model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
            reg_model.fit(X_train_reg, y_train_reg)
            reg_preds = reg_model.predict(X_test_reg)

            mae = mean_absolute_error(y_test_reg, reg_preds)
            r2 = r2_score(y_test_reg, reg_preds)

            importance_df = pd.DataFrame(
                {"Feature": X.columns, "Importance": reg_model.feature_importances_}
            ).sort_values("Importance", ascending=False)

            discounts["Is_Sale_Imminent"] = (discounts["Days Until Next Sale"] <= 18).astype(int)
            Y_class = discounts["Is_Sale_Imminent"]

            stratify_opt = Y_class if len(np.unique(Y_class)) > 1 else None
            X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
                X, Y_class, test_size=0.2, random_state=42, stratify=stratify_opt
            )

            clf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
            clf.fit(X_train_clf, y_train_clf)
            clf_preds = clf.predict(X_test_clf)

            cm = confusion_matrix(y_test_clf, clf_preds)
            cm_df = pd.DataFrame(
                cm,
                index=["Actual: No Sale", "Actual: Sale Soon"][:len(cm)],
                columns=["Pred: No Sale", "Pred: Sale Soon"][:len(cm[0])],
            )
            report = classification_report(y_test_clf, clf_preds, zero_division=0)


            target_info = fetch_single_game_data(target_appid)
            game_title = target_info.get("title", f"AppID {target_appid}").upper() if target_info else f"APPID {target_appid}"
            if print_test_bool:
                print(f"PERFORMANCE REPORT FOR {game_title}:")
                print(f"Total Sales Analyzed: {len(discounts)}")
                print(f"Mean Absolute Error: {mae:.2f} days")
                print(f"R^2 Score: {r2:.2f}")
                print("\n[Feature Importances]")
                print(importance_df.to_string(index=False))

                print("\n[Classification Accuracy (<=18 Days)]")
                print(cm_df)
                print(report)

            return X, reg_model, clf, feature_cols, raw_game_df, game_title

def predict_current_sale_status(X, reg_model, clf, feature_cols, raw_game_df, game_title):
    """Generates real-time sale predictions starting from today's date."""
    target_features = create_target_game_features(raw_game_df, feature_cols)
    if target_features is not None:
        predicted_days = reg_model.predict(target_features)[0]
        sale_imminent = clf.predict(target_features)[0]
        imminent_prob = clf.predict_proba(target_features)[0][1] if len(clf.classes_) == 2 else (1.0 if sale_imminent == 1 else 0.0)

        print(f"Predicted Days Until Next Sale: {predicted_days:.1f} days")
        print(f"Probability of a Sale within 18 Days: {imminent_prob*100:.1f}%")
    return


predict_current_sale_status(*prediction_function_single(1361510, False))

Predicted Days Until Next Sale: 32.1 days
Probability of a Sale within 18 Days: 42.0%
